### Import Dependencies

In [88]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [89]:
import torch
from torchvision import datasets, transforms, models  # datsets  , transforms
from torch.utils.data.sampler import SubsetRandomSampler
import torch.nn as nn
import torch.nn.functional as F
from datetime import datetime

In [90]:
%load_ext nb_black

ModuleNotFoundError: No module named 'nb_black'

### Import Dataset

<b> Dataset Link (Plant Vliiage Dataset ):</b><br> <a href='https://data.mendeley.com/datasets/tywbtsjrjv/1'> https://data.mendeley.com/datasets/tywbtsjrjv/1 </a> 

In [91]:
transform = transforms.Compose(
    [transforms.Resize(255), transforms.CenterCrop(224), transforms.ToTensor()]
)

In [92]:
dataset = datasets.ImageFolder("Dataset", transform=transform)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'Dataset'

In [ ]:
dataset

In [14]:
indices = list(range(len(dataset)))

NameError: name 'dataset' is not defined

In [15]:
split = int(np.floor(0.85 * len(dataset)))  # train_size

NameError: name 'dataset' is not defined

In [16]:
validation = int(np.floor(0.70 * split))  # validation

NameError: name 'split' is not defined

In [17]:
print(0, validation, split, len(dataset))

NameError: name 'validation' is not defined

In [18]:
print(f"length of train size :{validation}")
print(f"length of validation size :{split - validation}")
print(f"length of test size :{len(dataset)-validation}")

NameError: name 'validation' is not defined

In [19]:
np.random.shuffle(indices)

NameError: name 'indices' is not defined

### Split into Train and Test

In [20]:
train_indices, validation_indices, test_indices = (
    indices[:validation],
    indices[validation:split],
    indices[split:],
)

NameError: name 'indices' is not defined

In [21]:
train_sampler = SubsetRandomSampler(train_indices)
validation_sampler = SubsetRandomSampler(validation_indices)
test_sampler = SubsetRandomSampler(test_indices)

NameError: name 'train_indices' is not defined

In [22]:
targets_size = len(dataset.class_to_idx)

NameError: name 'dataset' is not defined

### Model

<b>Convolution Aithmetic Equation : </b>(W - F + 2P) / S + 1 <br>
W = Input Size<br>
F = Filter Size<br>
P = Padding Size<br>
S = Stride <br>

### Transfer Learning

In [23]:
# model = models.vgg16(pretrained=True)

In [24]:
# for params in model.parameters():
#     params.requires_grad = False

In [25]:
# model

In [26]:
# n_features = model.classifier[0].in_features
# n_features

In [27]:
# model.classifier = nn.Sequential(
#     nn.Linear(n_features, 1024),
#     nn.ReLU(),
#     nn.Dropout(0.4),
#     nn.Linear(1024, targets_size),
# )

In [28]:
# model

### Original Modeling

In [29]:
class CNN(nn.Module):
    def __init__(self, K):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            # conv1
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),
            # conv2
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),
            # conv3
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2),
            # conv4
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2),
        )

        self.dense_layers = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(50176, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, K),
        )

    def forward(self, X):
        out = self.conv_layers(X)

        # Flatten
        out = out.view(-1, 50176)

        # Fully connected
        out = self.dense_layers(out)

        return out

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [31]:
device = "cpu"

In [32]:
model = CNN(targets_size)

NameError: name 'targets_size' is not defined

In [33]:
model.to(device)

NameError: name 'model' is not defined

In [34]:
from torchsummary import summary

summary(model, (3, 224, 224))

ModuleNotFoundError: No module named 'torchsummary'

In [35]:
criterion = nn.CrossEntropyLoss()  # this include softmax + cross entropy loss
optimizer = torch.optim.Adam(model.parameters())

NameError: name 'model' is not defined

### Batch Gradient Descent

In [36]:
def batch_gd(model, criterion, train_loader, test_laoder, epochs):
    train_losses = np.zeros(epochs)
    validation_losses = np.zeros(epochs)

    for e in range(epochs):
        t0 = datetime.now()
        train_loss = []
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()

            output = model(inputs)

            loss = criterion(output, targets)

            train_loss.append(loss.item())  # torch to numpy world

            loss.backward()
            optimizer.step()

        train_loss = np.mean(train_loss)

        validation_loss = []

        for inputs, targets in validation_loader:

            inputs, targets = inputs.to(device), targets.to(device)

            output = model(inputs)

            loss = criterion(output, targets)

            validation_loss.append(loss.item())  # torch to numpy world

        validation_loss = np.mean(validation_loss)

        train_losses[e] = train_loss
        validation_losses[e] = validation_loss

        dt = datetime.now() - t0

        print(
            f"Epoch : {e+1}/{epochs} Train_loss:{train_loss:.3f} Test_loss:{validation_loss:.3f} Duration:{dt}"
        )

    return train_losses, validation_losses

In [37]:
device = "cpu"

In [38]:
batch_size = 64
train_loader = torch.utils.data.DataLoader(
    dataset, batch_size=batch_size, sampler=train_sampler
)
test_loader = torch.utils.data.DataLoader(
    dataset, batch_size=batch_size, sampler=test_sampler
)
validation_loader = torch.utils.data.DataLoader(
    dataset, batch_size=batch_size, sampler=validation_sampler
)

NameError: name 'dataset' is not defined

In [39]:
train_losses, validation_losses = batch_gd(
    model, criterion, train_loader, validation_loader, 5
)

NameError: name 'model' is not defined

### Save the Model

In [40]:
# torch.save(model.state_dict() , 'plant_disease_model_1.pt')

### Load Model

In [41]:
targets_size = 39
model = CNN(targets_size)
model.load_state_dict(torch.load("plant_disease_model_1_latest.pt"))
model.eval()

FileNotFoundError: [Errno 2] No such file or directory: 'plant_disease_model_1_latest.pt'

In [ ]:
# %matplotlib notebook

### Plot the loss

In [ ]:
plt.plot(train_losses , label = 'train_loss')
plt.plot(validation_losses , label = 'validation_loss')
plt.xlabel('No of Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

### Accuracy

In [42]:
def accuracy(loader):
    n_correct = 0
    n_total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model(inputs)

        _, predictions = torch.max(outputs, 1)

        n_correct += (predictions == targets).sum().item()
        n_total += targets.shape[0]

    acc = n_correct / n_total
    return acc

In [43]:
train_acc = accuracy(train_loader)
test_acc = accuracy(test_loader)
validation_acc = accuracy(validation_loader)

NameError: name 'train_loader' is not defined

In [44]:
print(
    f"Train Accuracy : {train_acc}\nTest Accuracy : {test_acc}\nValidation Accuracy : {validation_acc}"
)

NameError: name 'train_acc' is not defined

### Single Image Prediction

In [45]:
transform_index_to_disease = dataset.class_to_idx

NameError: name 'dataset' is not defined

In [46]:
transform_index_to_disease = dict(
    [(value, key) for key, value in transform_index_to_disease.items()]
)  # reverse the index

NameError: name 'transform_index_to_disease' is not defined

In [47]:
data = pd.read_csv("disease_info.csv", encoding="cp1252")

FileNotFoundError: [Errno 2] No such file or directory: 'disease_info.csv'

In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF

In [ ]:
def single_prediction(image_path):
    image = Image.open(image_path)
    image = image.resize((224, 224))
    input_data = TF.to_tensor(image)
    input_data = input_data.view((-1, 3, 224, 224))
    output = model(input_data)
    output = output.detach().numpy()
    index = np.argmax(output)
    print("Original : ", image_path[12:-4])
    pred_csv = data["disease_name"][index]
    print(pred_csv)

In [48]:
single_prediction("test_images/Apple_ceder_apple_rust.JPG")

NameError: name 'single_prediction' is not defined

### Wrong Prediction

In [49]:
single_prediction("test_images/Apple_scab.JPG")

NameError: name 'single_prediction' is not defined

In [50]:
single_prediction("test_images/Grape_esca.JPG")

NameError: name 'single_prediction' is not defined

In [51]:
single_prediction("test_images/apple_black_rot.JPG")

NameError: name 'single_prediction' is not defined

In [52]:
single_prediction("test_images/apple_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [53]:
single_prediction("test_images/background_without_leaves.jpg")

NameError: name 'single_prediction' is not defined

In [54]:
single_prediction("test_images/blueberry_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [55]:
single_prediction("test_images/cherry_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [56]:
single_prediction("test_images/cherry_powdery_mildew.JPG")

NameError: name 'single_prediction' is not defined

In [57]:
single_prediction("test_images/corn_cercospora_leaf.JPG")

NameError: name 'single_prediction' is not defined

In [58]:
single_prediction("test_images/corn_common_rust.JPG")

NameError: name 'single_prediction' is not defined

In [59]:
single_prediction("test_images/corn_healthy.jpg")

NameError: name 'single_prediction' is not defined

In [60]:
single_prediction("test_images/corn_northen_leaf_blight.JPG")

NameError: name 'single_prediction' is not defined

In [61]:
single_prediction("test_images/grape_black_rot.JPG")

NameError: name 'single_prediction' is not defined

In [62]:
single_prediction("test_images/grape_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [63]:
single_prediction("test_images/grape_leaf_blight.JPG")

NameError: name 'single_prediction' is not defined

In [64]:
single_prediction("test_images/orange_haunglongbing.JPG")

NameError: name 'single_prediction' is not defined

In [65]:
single_prediction("test_images/peach_bacterial_spot.JPG")

NameError: name 'single_prediction' is not defined

In [66]:
single_prediction("test_images/peach_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [67]:
single_prediction("test_images/pepper_bacterial_spot.JPG")

NameError: name 'single_prediction' is not defined

In [68]:
single_prediction("test_images/pepper_bell_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [69]:
single_prediction("test_images/potato_early_blight.JPG")

NameError: name 'single_prediction' is not defined

In [70]:
single_prediction("test_images/potato_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [71]:
single_prediction("test_images/potato_late_blight.JPG")

NameError: name 'single_prediction' is not defined

In [72]:
single_prediction("test_images/raspberry_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [73]:
single_prediction("test_images/soyaben healthy.JPG")

NameError: name 'single_prediction' is not defined

In [74]:
single_prediction("test_images/potato_late_blight.JPG")

NameError: name 'single_prediction' is not defined

In [75]:
single_prediction("test_images/squash_powdery_mildew.JPG")

NameError: name 'single_prediction' is not defined

In [76]:
single_prediction("test_images/starwberry_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [77]:
single_prediction("test_images/starwberry_leaf_scorch.JPG")

NameError: name 'single_prediction' is not defined

In [78]:
single_prediction("test_images/tomato_bacterial_spot.JPG")

NameError: name 'single_prediction' is not defined

In [79]:
single_prediction("test_images/tomato_early_blight.JPG")

NameError: name 'single_prediction' is not defined

In [80]:
single_prediction("test_images/tomato_healthy.JPG")

NameError: name 'single_prediction' is not defined

In [81]:
single_prediction("test_images/tomato_late_blight.JPG")

NameError: name 'single_prediction' is not defined

In [82]:
single_prediction("test_images/tomato_leaf_mold.JPG")

NameError: name 'single_prediction' is not defined

In [83]:
single_prediction("test_images/tomato_mosaic_virus.JPG")

NameError: name 'single_prediction' is not defined

In [84]:
single_prediction("test_images/tomato_septoria_leaf_spot.JPG")

NameError: name 'single_prediction' is not defined

In [85]:
single_prediction("test_images/tomato_spider_mites_two_spotted_spider_mites.JPG")

NameError: name 'single_prediction' is not defined

In [86]:
single_prediction("test_images/tomato_target_spot.JPG")

NameError: name 'single_prediction' is not defined

In [87]:
single_prediction("test_images/tomato_yellow_leaf_curl_virus.JPG")

NameError: name 'single_prediction' is not defined